# Conversion Unimarc vers matrice excel

Import :
- le script SRUextractionMODIFIE.py (version Mathias) ou SRUextraction.py (version initiale) est nécessaire pour interroger le SRU BnF lorsqu'un ISNI n'est pas trouvé dans le mapping local ;
- le fichier Numelyo-mapping-isnis-nnas.txt doit être présent dans le répertoire de travail du notebook ;
- les lignes d'installation ci-dessous permettent de compléter les bibliothèques manquantes dans l'environnement Jupyter.


In [1]:
# Interrogation du SRU BnF : version modifiée si disponible, sinon version initiale
try:
    import SRUextractionMODIFIE as sru
except ModuleNotFoundError as e:
    if e.name != "SRUextractionMODIFIE":
        raise
    import SRUextraction as sru

try:
    import pandas
except ModuleNotFoundError:
    %pip install pandas
    import pandas

import re
from collections import Counter

try:
    from pymarc import MARCReader, Field
except ModuleNotFoundError:
    %pip install pymarc
    from pymarc import MARCReader, Field

try:
    from tqdm import tqdm
except ModuleNotFoundError:
    %pip install tqdm
    from tqdm import tqdm

try:
    from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
except ModuleNotFoundError:
    %pip install openpyxl
    from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE


### Ajout du mapping ISNI-NNA
On cherche d'abord le NNA dans le fichier de correspondances fourni par la BML.
**Si l'ISNI n'est pas présent dans le mapping, ou si son NNA est vide, on interroge le SRU de la BnF.**
Les erreurs de requêtage sont conservées dans `ERREURS_SRU`, sans interrompre la conversion des autres notices.


In [2]:
import csv
from pathlib import Path

MAPPING_PATH = Path("Numelyo-mapping-isnis-nnas.txt")

# Les ISNI peuvent être saisis avec ou sans le préfixe "ISNI" dans les notices Unimarc
# (ex. 700$o0000000083509777 ou 700$oISNI0000000083509777).
def normaliser_isni(value):
    value = re.sub(r"^ISNI\s*:?\s*", "", str(value or "").strip(), flags=re.I)
    value = re.sub(r"[\s-]", "", value).upper()
    return value if re.fullmatch(r"\d{15}[\dX]", value) else ""

ISNI_TO_NNA = {}

with MAPPING_PATH.open(encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")

    for row in reader:
        raw_isni = (row.get("ISNI") or "").strip().upper()
        nna = (row.get("NNA") or "").strip()

        # Certaines lignes ajoutent le nom de la collectivité après les 16 caractères de l'ISNI
        match = re.match(r"^(\d{15}[\dX])(?=\s|$)", raw_isni)
        if not match or not re.fullmatch(r"\d{8}", nna):
            continue

        isni = match.group(1)
        if isni in ISNI_TO_NNA and ISNI_TO_NNA[isni] != nna:
            raise ValueError(f"Correspondances NNA contradictoires pour l'ISNI {isni}")
        ISNI_TO_NNA[isni] = nna

# Le cache évite plusieurs interrogations SRU pour un même ISNI absent du mapping
SRU_CACHE_NNA = {}
ERREURS_SRU = {}

def recuperer_nna(isni):
    if not isni:
        return ""

    # Priorité au mapping BML, puis interrogation du SRU BnF en cas d'absence
    nna = ISNI_TO_NNA.get(isni, "")
    if nna:
        return nna
    if isni in SRU_CACHE_NNA:
        return SRU_CACHE_NNA[isni]

    nna = ""
    try:
        result = sru.SRU_result(
            f"aut.isni any '{isni}'",
            parametres={"recordSchema": "intermarcxchange"}
        )
        record_aut = result.firstRecord if result.test else None
        if record_aut is not None:
            value = sru.record2fieldvalue(record_aut, "001")
            value = str(value or "")

            # Dans les premiers scripts, le NNA est extrait de 001 par [5:].
            # Certaines formes possèdent en plus un caractère de contrôle final : [5:-1].
            candidat = value[5:]
            if re.fullmatch(r"\d{8}", candidat):
                nna = candidat
            elif re.fullmatch(r"\d{8}[0-9A-Za-z]", candidat):
                nna = candidat[:-1]
            elif value:
                ERREURS_SRU[isni] = f"Format 001 à contrôler : {value!r}"
    except Exception as e:
        # Garder la notice même si le service SRU n'est pas disponible
        ERREURS_SRU[isni] = str(e)

    SRU_CACHE_NNA[isni] = nna
    return nna

print(f"Mapping chargé : {len(ISNI_TO_NNA)} correspondances")


Mapping chargé : 1066 correspondances


### Référentiels
Ici, on commence par prendre les référentiels intermarc NG. 
Dans le cas des codes fonctions et des catégories techniques, les informations sont exprimées de façon différentes en Unimarc et en Intermarc NG : les dictionnaires ont pour but de permettre leur conversion.
Attention, ces dictionnaires ne sont pas exhaustifs : les seuls cas de figures pris en compte ici sont ceux explicitement discutés avec la BML. S'il y a utilisation d'autres codes fonctions ou d'autres informations en 200$b, la conversion ne sera pas satisfaisante.

In [3]:
CODES_PAYS = ["##", "ad", "ae", "af", "ag", "ai", "al", "am", "ao", "aq", "ar", "as", "at", "au", "aw", "ax", "az", "ba", "bb", "bd", "be", "bf", "bf", "bf", "bg", "bh", "bi", "bj", "bl", "bm", "bn", "bo", "bq", "bq", "bq", "br", "bs", "bt", "bv", "bw", "by", "by", "bz", "ca", "cc", "cc", "cd", "cd", "cf", "cf", "cf", "cg", "cg", "ch", "ch", "ci", "ck", "cl", "cm", "cn", "cn", "co", "cr", "cu", "cv", "cw", "cx", "cy", "cz", "cz", "de", "de", "dj", "dk", "dm", "do", "do", "dz", "ec", "ee", "eg", "eh", "er", "es", "et", "eu", "fi", "fj", "fk", "fk", "fm", "fm", "fo", "fr", "ga", "gb", "gb", "gd", "ge", "gf", "gg", "gh", "gi", "gl", "gm", "gn", "gp", "gq", "gr", "gs", "gt", "gu", "gw", "gy", "hk", "hm", "hn", "hr", "ht", "hu", "id", "ie", "il", "im", "im", "in", "io", "io", "io", "iq", "ir", "is", "it", "je", "jm", "jo", "jp", "ke", "kg", "kh", "ki", "km", "kn", "kn", "kp", "kp", "kr", "kr", "kw", "ky", "ky", "ky", "kz", "la", "lb", "lc", "li", "lk", "lr", "ls", "lt", "lu", "lv", "ly", "ly", "ma", "mc", "md", "md", "me", "mf", "mg", "mh", "mk", "ml", "mm", "mm", "mn", "mo", "mp", "mq", "mr", "ms", "mt", "mu", "mv", "mw", "mx", "my", "mz", "na", "nc", "ne", "nf", "ng", "ni", "nl", "no", "np", "nr", "nu", "nz", "om", "pa", "pe", "pf", "pg", "ph", "pk", "pl", "pm", "pn", "pr", "ps", "ps", "pt", "pw", "pw", "py", "qa", "re", "ro", "rs", "ru", "ru", "rw", "sa", "sb", "sc", "sd", "se", "sg", "sh", "sh", "sh", "si", "sj", "sk", "sl", "sm", "sn", "so", "sr", "ss", "ss", "st", "sv", "sv", "sx", "sx", "sy", "sz", "tc", "td", "tf", "tg", "th", "tj", "tk", "tl", "tm", "tn", "to", "tr", "tt", "tv", "tw", "tw", "tz", "ua", "ug", "um", "us", "uy", "uz", "va", "va", "vc", "ve", "vg", "vg", "vi", "vi", "vn", "vu", "wf", "ws", "ws", "xk", "ye", "ye", "yt", "za", "zm", "zw", "zz"]


In [5]:
CODES_LANGUE = ["###", "aar", "abk", "ace", "ace", "ach", "ach", "ach", "ada", "ada", "ada", "ada", "ady", "afa", "afa", "afh", "afr", "ain", "aka", "aka", "aka", "akk", "akk", "akk", "alb", "alb", "ale", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alt", "amh", "ang", "ang", "ang", "ang", "anp", "apa", "apa", "ara", "arc", "arc", "arc", "arg", "arm", "arn", "arn", "arn", "arp", "art", "art", "art", "arw", "asm", "ast", "ast", "ast", "ast", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "aus", "aus", "aus", "aus", "aus", "aus", "aus", "ava", "ave", "ave", "ave", "ave", "ave", "ave", "awa", "aym", "aze", "aze", "bad", "bad", "bai", "bai", "bak", "bak", "bal", "bam", "bam", "ban", "baq", "baq", "baq", "bas", "bas", "bas", "bas", "bas", "bas", "bat", "bat", "bej", "bej", "bej", "bel", "bem", "ben", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "bho", "bih", "bih", "bik", "bin", "bin", "bis", "bis", "bis", "bla", "bla", "bla", "bnt", "bnt", "bnt", "bnt", "bos", "bra", "bra", "bre", "bre", "bre", "bre", "bre", "bre", "btk", "btk", "bua", "bug", "bul", "bur", "byn", "byn", "byn", "cad", "cad", "cad", "cai", "cai", "car", "car", "car", "car", "cat", "cat", "cau", "cau", "cau", "cau", "cau", "cau", "ceb", "cel", "cel", "cel", "cel", "cel", "cel", "cha", "chb", "che", "chg", "chg", "chg", "chi", "chk", "chk", "chm", "chn", "chn", "chn", "chn", "cho", "chp", "chr", "chu", "chu", "chu", "chu", "chu", "chu", "chv", "chy", "cmc", "cmc", "cop", "cor", "cos", "cpe", "cpe", "cpe", "cpe", "cpe", "cpe", "cpe", "cpe", "cpf", "cpf", "cpf", "cpf", "cpf", "cpf", "cpf", "cpf", "cpf", "cpf", "cpp", "cpp", "cpp", "cpp", "cpp", "cre", "cre", "cre", "crh", "crh", "crp", "crp", "crp", "crp", "crp", "crp", "crp", "csb", "cus", "cus", "cus", "cus", "cze", "dak", "dak", "dak", "dak", "dak", "dak", "dak", "dan", "dar", "day", "day", "del", "den", "den", "dgr", "dgr", "dgr", "din", "din", "div", "doi", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dsb", "dua", "dum", "dum", "dum", "dut", "dut", "dyu", "dyu", "dyu", "dzo", "dzo", "efi", "efi", "egy", "egy", "egy", "egy", "eka", "elx", "elx", "elx", "elx", "elx", "elx", "elx", "eng", "enm", "enm", "enm", "epo", "est", "ewe", "ewe", "ewe", "ewo", "fan", "fan", "fao", "fat", "fij", "fil", "fil", "fin", "fin", "fin", "fiu", "fiu", "fiu", "fiu", "fiu", "fon", "frdbnf", "frdbnf", "fre", "frm", "frm", "frm", "fro", "fro", "fro", "fro", "frr", "frs", "fry", "ful", "ful", "ful", "ful", "ful", "fur", "gaa", "gay", "gba", "gedbnf", "gedbnf", "gem", "gem", "gem", "gem", "gem", "gem", "geo", "ger", "gez", "gez", "gez", "gez", "gez", "gez", "gil", "gil", "gla", "gla", "gla", "gla", "gle", "gle", "gle", "gle", "glg", "glv", "glv", "glv", "gmh", "gmh", "goh", "goh", "gon", "gon", "gor", "got", "grb", "grc", "grc", "grc", "grc", "grc", "grc", "grc", "grc", "grdbnf", "grdbnf", "gre", "gre", "grm", "grn", "grp", "gsw", "gsw", "gsw", "gsw", "gsw", "gsw", "guj", "guj", "guj", "gwi", "hai", "hat", "hat", "hat", "hau", "haw", "heb", "heb", "her", "hil", "him", "him", "him", "hin", "hit", "hmn", "hmo", "hrv", "hsb", "hsb", "hun", "hun", "hup", "iba", "ibo", "ibo", "ice", "ido", "iii", "ijo", "ijo", "ijo", "ijo", "iku", "ile", "ilo", "ina", "ina", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "ind", "ind", "ine", "ine", "ine", "ine", "ine", "ine", "ine", "inh", "ipk", "ira", "ira", "ira", "ira", "ira", "ira", "ira", "ira", "iro", "iro", "iro", "iro", "iro", "iro", "iro", "iro", "ita", "itdbnf", "itdbnf", "itdbnf", "itdbnf", "jav", "jbo", "jpn", "jpr", "jrb", "kaa", "kab", "kab", "kac", "kac", "kal", "kal", "kam", "kan", "kan", "kan", "kan", "kar", "kar", "kas", "kas", "kas", "kas", "kau", "kaw", "kaw", "kaw", "kaz", "kbd", "kha", "khi", "khi", "khi", "khi", "khi", "khm", "khm", "kho", "kho", "kho", "kik", "kin", "kin", "kir", "kmb", "kmb", "kok", "kom", "kon", "kon", "kor", "kos", "kpe", "kpe", "kpe", "krc", "krl", "kro", "kro", "kro", "kro", "kro", "kro", "kro", "kru", "kua", "kua", "kum", "kur", "kut", "kut", "lad", "lad", "lad", "lah", "lah", "lah", "lam", "lao", "lao", "lat", "lav", "lav", "lav", "lez", "lim", "lin", "lit", "lol", "lol", "lol", "loz", "lsf", "ltz", "lua", "lub", "lub", "lub", "lug", "lug", "lui", "lun", "luo", "luo", "luo", "lus", "mac", "mad", "mag", "mag", "mag", "mag", "mah", "mai", "mak", "mal", "man", "man", "man", "man", "mao", "map", "map", "map", "map", "map", "map", "mar", "mar", "mas", "mas", "may", "mdf", "mdr", "men", "mga", "mga", "mga", "mga", "mic", "mic", "min", "min", "mis", "mis", "mkh", "mkh", "mlg", "mlt", "mnc", "mni", "mni", "mno", "mno", "moh", "mon", "mos", "mos", "mun", "mun", "mun", "mun", "mun", "mun", "mun", "mun", "mun", "mus", "mus", "mwl", "mwr", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myv", "nah", "nah", "nah", "nah", "nah", "nai", "nai", "nai", "nap", "nau", "nav", "nbl", "nde", "ndo", "ndo", "nds", "nds", "nds", "nds", "nep", "new", "new", "new", "nia", "nic", "nic", "nic", "nic", "nic", "niu", "niu", "nno", "nno", "nno", "nno", "nob", "nob", "nog", "nog", "non", "non", "non", "non", "non", "non", "non", "non", "nor", "nqo", "nso", "nso", "nso", "nub", "nub", "nub", "nub", "nub", "nub", "nwc", "nya", "nya", "nya", "nya", "nya", "nym", "nyn", "nyn", "nyo", "nzi", "nzi", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oji", "oji", "oji", "oji", "ori", "orm", "orm", "osa", "oss", "ota", "ota", "ota", "ota", "oto", "oto", "oto", "oto", "oto", "oto", "oto", "oto", "paa", "paa", "paa", "pag", "pal", "pal", "pal", "pal", "pal", "pam", "pan", "pan", "pan", "pap", "pau", "pau", "peo", "peo", "per", "per", "per", "per", "per", "phi", "phi", "phn", "pli", "pol", "pon", "por", "pra", "pra", "pro", "pro", "pro", "pus", "pus", "pus", "pus", "que", "que", "que", "raj", "rap", "rar", "rar", "roa", "roa", "roa", "roa", "roa", "roh", "roh", "roh", "rom", "rom", "rom", "rom", "rom", "rum", "rum", "run", "run", "run", "rup", "rup", "rus", "sad", "sag", "sah", "sah", "sah", "sai", "sai", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sam", "san", "san", "sas", "sat", "scn", "sco", "sco", "sel", "sel", "sem", "sem", "sem", "sem", "sem", "sem", "sem", "sem", "sem", "sem", "sem", "sga", "sga", "sga", "sgn", "shn", "shn", "sid", "sin", "sin", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sit", "sit", "sit", "sit", "sit", "sit", "sit", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "slo", "slv", "sma", "sme", "smi", "smi", "smi", "smi", "smj", "smn", "smo", "sms", "sna", "sna", "snd", "snk", "sog", "som", "son", "son", "sot", "sot", "sot", "sot", "spa", "spa", "spdbnf", "spdbnf", "spdbnf", "srd", "srn", "srp", "srr", "srr", "ssa", "ssa", "ssa", "ssa", "ssa", "ssw", "ssw", "ssw", "suk", "sun", "sun", "sus", "sus", "sus", "sux", "swa", "swa", "swe", "swe", "syc", "syr", "syr", "tah", "tai", "tai", "tai", "tai", "tai", "tai", "tai", "tai", "tai", "tai", "tai", "tam", "tam", "tat", "tel", "tel", "tel", "tem", "tem", "tem", "ter", "ter", "tet", "tgk", "tgl", "tgl", "tha", "tha", "tib", "tig", "tir", "tir", "tiv", "tiv", "tiv", "tkl", "tlh", "tli", "tmh", "tmh", "tog", "tog", "ton", "tpi", "tsi", "tsi", "tsn", "tsn", "tso", "tuk", "tuk", "tum", "tup", "tup", "tur", "tut", "tut", "tut", "tut", "tut", "tut", "tvl", "twi", "twi", "tyv", "udm", "udm", "udm", "uga", "uga", "uig", "ukr", "umb", "umb", "umb", "urd", "urd", "uzb", "uzb", "vai", "vai", "vai", "ven", "vie", "vie", "vol", "vot", "wak", "wak", "wak", "wak", "wak", "wak", "wak", "wak", "wal", "wal", "wal", "war", "was", "wel", "wel", "wen", "wen", "wen", "wln", "wol", "wol", "wol", "xal", "xal", "xho", "xho", "xho", "yao", "yap", "yap", "yid", "yid", "yor", "ypk", "ypk", "zap", "zbl", "zbl", "zen", "zha", "zha", "znd", "znd", "zul", "zun", "zza", "zza", "zza", "zza", "zza", "zza"]

In [6]:
# Correspondances spécifiques validées avec la BML
CODES_LANGUE_BML = {"deu": "ger", "esp": "spa", "nld": "dut"}
CODES_PAYS_BML = {"DX": "de", "SP": "es", "XX": "xx", "YU": "yucs"}

# "xx" et "yucs" font partie des valeurs attendues dans le livrable,
# mais ne sont pas présents dans la liste de pays initiale.
CODES_PAYS_VALIDES = set(CODES_PAYS) | {"xx", "yucs"}
CODES_LANGUE_VALIDES = set(CODES_LANGUE) | {"und"}


In [7]:
CODES_FONCTION = {"040": "0990", # artiste : créateur / créatrice de l'œuvre
                  "650": "3250", # éditeur commercial (cartes postales)
                  "560": "0190", # commanditaire (affiches, photographies)
                  "570": "9990",
                  "070" : "0990",
                  "440" : "0440",
                  "260": "4210",
                  "390": "4010",
                  "320": "4040",
                  "340" : "9990",
                  "350": "0410",
                  "360" : "0410",
                  "530" : "0410",
                  "760" : "0410",
                  "510" : "0480",
                  "600" : "0530",
                  "610" : "3060",
                  "620" : "3060",
                  "740" : "2310",
                  "750": "2310",
                  "" : "" # avant on mettait 9990
                 }
                 
# Correspondances des codes fonctions 040, 650 et 560 validées par la BML


Référentiel catégorie technique : ici, on utilise les valeurs renseignées en 200\\$b (saisie manuelle) pour alimenter le référentiel "catégorie technique" de la BnF de façon minimale. Si la 200\\$b n'a pas été correctement saisie, cela risque de créer des anomalies

In [8]:
CATEGORIE_TECHNIQUE = {
    "Photographie": "photographie",
    "Carte postale": "procédé photomécanique",
    "Affiche": "procédé photomécanique",
    "Estampe": "estampe",
    "Manuscrit": "autre catégorie technique" # relevés d'inscriptions (monuments funéraires / architecturaux)
}

# Conversion après suppression éventuelle des crochets entourant 200$b
CATEGORIE_TECHNIQUE_NORMALISE = {k.casefold(): v for k, v in CATEGORIE_TECHNIQUE.items()}


### Création d'une classe records pour les notices unimarcs

#### Fonctions utilisées pour créer la classe records

In [9]:
def field_to_string(field):
    
    res = ""
    subfields = field.subfields
    for subfield in subfields:
        res += f"${subfield.code} {subfield.value} "
    
    return res

In [10]:
def note_exe2string(zone_note, string):
    for note in zone_note:
        for subfield in note.get_subfields("a"):
            string += subfield if subfield.endswith(".") else subfield + "."
           
    return string
    

#### Passer l'indexation Rameau en chaîne de caractères
Le but des deux fonctions suivantes est de permettre de passer d'une indexation Rameau en unimarc à des chaînes de caractères, qu'on chargera en indexation non contrôlée en Intermarc NG

In [11]:
def liste_subfields(field):
    liste_subfields = []
    for e in field.split("$"):
        if e:
            liste_subfields.append((e[0], e[1:].strip()))
    return liste_subfields

def marcfield2label(field):
    if field:
        repetitions = {"a": " -- ",
                    "x": " -- ",
                    "y": " -- ",
                    "l": " / ",
                    "d": " ; "}
        for char in repetitions:
            if re.search(f"\${char}([^$]+) \${char}", field) is not None:
                field = re.sub(f"\${char}([^$]+) \${char}", f"${char}\\1 {repetitions[char]}", field)
            if re.search(f"\${char}([^$]+) \${char}", field) is not None:
                field = re.sub(f"\${char}([^$]+) \${char}", f"${char}\\1 {repetitions[char]}", field)
        rules_multi = {"d k j": "(d-k-j)",
                       "f e": "f e"}  # date précise
        rules = {"a": "a",
                "b": ". b",
                "c": " c",
                "x": " -- x",
                "y": " -- y",
                "5": " -- 5",
                "i": " (i)",
                "k": "-k",
                "j": "-j",
                "d": " d",
                "h": " h",
                "t": " t",
                "l": " (l)",
                "m": ". m",
                "n": ". n",
                "z": " -- z",
                "f": " (f)",
                "v": " (v)"
        }
        for rule in rules_multi:
            chars = rules_multi[rule].split(" ")
            if len(chars) == 2:
                if re.search(f"\${chars[0]}([^$]+) \${chars[1]}([^$]+)", field) is not None:
                    field = re.sub(f"\${chars[0]}([^$]+) \${chars[1]}([^$]+)", r"(\1 ; \2)", field)
            if len(chars) == 3:
                if re.search(f"\${chars[0]}([^$]+) \${chars[1]}([^$]+) \${chars[2]}([^$]+)", field) is not None:
                    field = re.sub(f"\${chars[0]}([^$]+) \${chars[1]}([^$]+) \${chars[2]}([^$]+)( ?)", r"(\1-\2-\3)\4", field)
        field = liste_subfields(field)

        i = 0
        
        for subfield in field:
            try:
                rule = rules[subfield[0]]
            except KeyError:
                rule = f" -- {subfield[0]}"
            
            if ("(" in rule and "(" in subfield[1]) or (")" in rule and ")" in subfield[1]):
                rule = f" {subfield[0]}"
            # print(70, re.sub(f"\${char}([^$]+)", rule.replace(subfield[0], subfield[1]), subfield[1]))
            field[i] = rule.replace(subfield[0], subfield[1])
            
            i += 1
        field = "".join(field)
        field = field.replace("( ", "(").replace(" )", ")")
        field = field.replace(")(", ' ; ')
        field = field.replace(" ;; ", ' ; ').replace(" ; ; ", ' ; ')
        field = field.replace(";)", ')')
        field = field.replace("((", '(').replace("))", ")")
        while "  " in field:
            field = field.replace("  ", " ")
        if re.search("\$[\w]", field) is not None:
            field = re.sub("\$[\w]", " ¤¤ ", field)
    field = field.strip()
    
    return field

#### Classe Auteur
On utilise d'abord le mapping ISNI-NNA validé par la BML, puis le SRU de la BnF pour les ISNI absents du fichier ou sans NNA.
**Le format exact du NNA extrait du champ SRU `001` doit être validé avec le SRE / BnF.**


In [12]:
class Auteur:
    """A partir d'une zone unimarc de type 7XX, on définit les attributs à intégrer dans les colonnes Auteur de la matrice.
    On en fait un objet à part pour garder ensemble les informations liées à un même auteur."""

    def __init__(self, field):

        self.tag = field.tag if field else ""
        self.field = field if field else ""
        self.nom = next(iter(field.get_subfields("a")), "") if field else ""
        self.dol_o = next(iter(field.get_subfields("o")), "") if field else ""

        # Normalisation de l'ISNI et recherche du NNA (mapping local, puis SRU si nécessaire)
        self.isni = normaliser_isni(self.dol_o)
        self.nna = recuperer_nna(self.isni)

        # La matrice ne gère pas les codes fonction répétés, on ne prend donc que le premier
        self.fonction = next(iter(field.get_subfields("4")), "") if field else ""

        # Conversion des codes fonction Unimarc
        self.fonction = CODES_FONCTION.get(self.fonction, "")
        # Si un code fonction n'est pas dans la liste prévue, il devient indéterminé par défaut
        if self.fonction == "" and self.nom != "":
            self.fonction = "" # "9990" 
            # il n’est pas nécessaire de renseigner des valeurs par défaut pour les codes langue et les codes fonction 
            # lorsque ces informations sont absentes des données d’origine. 

        # Nature : Collectivité si 710/711/712 ; Personne si 700/701/702/720/721/722
        self.type = ""
        if self.tag in ["710", "711", "712"]:
            self.type = "Collectivité"
        elif self.tag in ["700", "701", "702", "720", "721", "722"]: # 721 & 722 sortis du chapeau, mais ce sont des auteurs théoriquement! (cf classe notice_unimarc)
            self.type = "Personne"

        self.prenom = ""
        if self.type == "Collectivité":
            self.prenom = next(iter(field.get_subfields("b")), "") if field else ""
        elif self.type == "Personne":
            self.prenom = next(iter(field.get_subfields("g")), "") if field else ""
            if self.prenom == "":
                self.prenom = next(iter(field.get_subfields("b")), "") if field else ""


## Codes projet

On ajoute les codes projet dans les colonnes I71-01, I71-02 et I71-03. ATTENTION : ON A PLUS DE 4 CODES PROJETS DANS CERTAINS FICHIERS COMME GMB_FICHIER10. ON A PRIS LA DÉCISION D'AJOOUTER LES COLONNES I171-04, I-171-05 ET AINSI DE SUITE SI NÉCESSAIRE

In [13]:
# On importe le module unicodedata 
# pour que nous ne perdions pas les accents éventuels lors de la lecture 
# genre Dürenmatt/Durenmatt
import unicodedata

def normaliser_texte(valeur):
    """En normalisant : casse, accents et espaces."""
    valeur = unicodedata.normalize("NFKD", str(valeur or ""))
    valeur = "".join(
        c for c in valeur
        if not unicodedata.combining(c)
    )
    return " ".join(valeur.casefold().split())


def valeurs_unimarc(record, tag, code):
    """Nous récuperons ici toutes les occurrences d'une sous-zone."""
    return [
        normaliser_texte(v)
        for champ in record.get_fields(tag)
        for v in champ.get_subfields(code)
        if v.strip()
    ]


def attribuer_codes_projet(record):
    """Attribution des codes projet spécifiques BmL."""

    codes = ["LyoNu000"]

    # Je supprime autour des types de documents sinon le code ne détecte pas les correspondances ici
    types = [
        valeur.strip("[] ").strip()
        for valeur in valeurs_unimarc(record, "200", "b")
    ]
    fonds = valeurs_unimarc(record, "317", "a")
    series = valeurs_unimarc(record, "461", "t")

    def type_est(texte):
        return normaliser_texte(texte) in types

    def dans_fonds(texte):
        texte = normaliser_texte(texte)
        return any(texte in v for v in fonds)

    def dans_series(texte):
        texte = normaliser_texte(texte)
        return any(texte in v for v in series)

    def fonds_ou_serie(texte):
        return dans_series(texte) or dans_fonds(texte)

    def ajouter(code, condition):
        if condition and code not in codes:
            codes.append(code)

    # Typologie : 200$b
    ajouter("LyoNu016", type_est("Affiche"))
    ajouter("LyoNu017", type_est("Estampe"))
    ajouter("LyoNu018", type_est("Photographie"))
    ajouter("LyoNu019", type_est("Carte postale"))

    # Patrimoine arménien : 317
    ajouter(
        "LyoNu020",
        dans_fonds(
            "Dépôt de l’Union Internationale "
            "des Organisations Terre et Culture"
        ) or dans_fonds("UIOTC")
    )

    # Première Guerre mondiale : 317 ET 200$b
    guerre = dans_fonds("Fonds Guerre 14-18")

    ajouter("LyoNu021", guerre and type_est("Photographie"))
    ajouter("LyoNu022", guerre and type_est("Affiche"))
    ajouter("LyoNu023", guerre and type_est("Carte postale"))

    # Fonds et collections : 317 OU 461$t
    ajouter("LyoNu024", fonds_ou_serie("CRDP de Lyon"))
    ajouter("LyoNu025", fonds_ou_serie("Didier Nicole"))
    ajouter("LyoNu026", fonds_ou_serie("Alfred Reverdy"))
    ajouter("LyoNu027", fonds_ou_serie("Georges Vermard"))

    ajouter(
        "LyoNu028",
        dans_fonds("Maison du fleuve Rhône")
    )

    ajouter("LyoNu029", dans_series("Guy Dürrenmatt"))

    ajouter(
        "LyoNu030",
        dans_fonds("Jules Sylvestre")
    )

    ajouter("LyoNu031", dans_series("Lyon Figaro"))

    ajouter(
        "LyoNu032",
        dans_fonds("Marcelle Vallet")
    )

    ajouter(
        "LyoNu033",
        dans_fonds("Pierre Pallud")
    )

    ajouter("LyoNu034", dans_series("Vos photos"))
    ajouter("LyoNu035", dans_series("Pierre Rondeau"))

    # Cartes postales de la région
    series_cartes = (
        "Vos cartes postales",
        "Cartes postales DL",
        "Pierre Rondeau",
        "Guy Dürrenmatt",
        "Pierre Pallud",
    )

    ajouter(
        "LyoNu036",
        any(dans_series(nom) for nom in series_cartes)
    )

    ajouter("LyoNu037", dans_fonds("Josette Vial"))
    ajouter("LyoNu038", dans_fonds("Jacques-Joseph Dutey"))

    # LyoNu039 : Photographes en Auvergne-Rhône-Alpes
    # Attribué si au moins un code de LyoNu024 à LyoNu038
    # est présent parmi les codes projet du document.

    codes_collections_vertes = {
        f"LyoNu{i:03d}" for i in range(24, 39)
    }

    ajouter(
        "LyoNu039",
        bool(codes_collections_vertes.intersection(codes))
    )

    # ON VERIFIE : Maximum trois codes projet par notice
    # if len(codes) > 3:
        # raise ValueError(
            # f"Plus de 3 codes projet pour une notice : {codes}"
        # )

    return codes


#### Classe notice unimarc
Cette classe permet de constituer un objet avec les données à intégrer dans la matrice.
Les commentaires en français précisent les choix de conversion et les points restant à confirmer avec la BML / BnF.


In [14]:
def insert (source_str, insert_str, pos):
    return source_str[:pos] + insert_str + source_str[pos:]

# === AJOUT : INDEXATIONS ARMÉNIENNES ===

def contient_armenien(texte):
    """Détecte la présence de caractères arméniens."""
    return bool(re.search(r"[\u0530-\u058F\uFB13-\uFB17]", texte or ""))


def extraire_indexations_armeniennes(marc_record):
    """Récupère les indexations arméniennes sans limiter les occurrences."""

    sujets = []
    lieux = []

    zones_sujets = ("600", "601", "602", "604", "605",
                    "606", "608", "610")
    zones_lieux = ("607", "617")

    for tag in zones_sujets + zones_lieux:
        for champ in marc_record.get_fields(tag):
            ## MODIFICATION : Les valeurs arméniennes utilisent la ponctuation BnF.
            valeur = formater_indexation(
                champ, field_to_string, marcfield2label
            )

            # Supprimer le point superflu au début
            valeur = re.sub(r"^\s*\.\s*", "", valeur)

            if not valeur or not contient_armenien(valeur):
                continue

            destination = lieux if tag in zones_lieux else sujets

            if valeur not in destination:
                destination.append(valeur)

    return sujets, lieux


# === FIN AJOUT ===

class notice_unimarc:
    
   
    def __init__(self, marc_record):
        self.marc_record = marc_record
        
        
       
        self.provenance = "BM de Lyon"
       
        self.num_notice = marc_record["001"].data
        self.ark = ""
        """ Attention ! Retier dièse sur la ligne suivante pour récupérer l'ark de la notice (absent de l'échantillon, mais présent dans le lot final)"""
        self.ark = marc_record["003"].data if marc_record["003"] else "" # Ça a changé ça, mais normalement pas de souci

        """Auteurs"""      
        champs_prioritaires = ["700", "710", "720", "701", "711", "721", "702", "712", "722"]
        auteurs = []
        for tag in champs_prioritaires:
            for field in marc_record.get_fields(tag):
                auteur = Auteur(field) if field else None
                if auteur and auteur not in auteurs and len(auteurs) <3:
                        auteurs.append(auteur)

        self.Auteur1 = auteurs[0] if len(auteurs)>0 else Auteur(None)
        self.Auteur2 = auteurs[1] if len(auteurs)>1 else Auteur(None)
        self.Auteur3 = auteurs[2] if len(auteurs)>2 else Auteur(None)
        
    
        
        
        """Langues"""
        self.liste_langues =  marc_record.get_fields("101")[0].get_subfields("a") if len(marc_record.get_fields("101"))>0 else []
        self.langue1 = self.liste_langues[0] if len(self.liste_langues) > 0 else None
        self.langue2 = self.liste_langues[1] if len(self.liste_langues) > 1 else None
        self.langue3 = self.liste_langues[2] if len(self.liste_langues) > 2 else None
        # Conversion des codes langue spécifiques à la BML (deu/esp/nld)
        self.langue1 = CODES_LANGUE_BML.get((self.langue1 or "").lower(), (self.langue1 or "").lower())
        self.langue2 = CODES_LANGUE_BML.get((self.langue2 or "").lower(), (self.langue2 or "").lower())
        self.langue3 = CODES_LANGUE_BML.get((self.langue3 or "").lower(), (self.langue3 or "").lower())
        if self.langue1 not in CODES_LANGUE_VALIDES:
            self.langue1 = "" # "und"
            # il n’est pas nécessaire de renseigner des valeurs par défaut pour les codes langue et les codes fonction 
            # lorsque ces informations sont absentes des données d’origine.
        if self.langue2 and self.langue2 not in CODES_LANGUE_VALIDES:
            self.langue2 = "" # "und"
            # il n’est pas nécessaire de renseigner des valeurs par défaut pour les codes langue et les codes fonction 
            # lorsque ces informations sont absentes des données d’origine.
        if self.langue3 and self.langue3 not in CODES_LANGUE_VALIDES:
            self.langue3 = "" # "und"
            # il n’est pas nécessaire de renseigner des valeurs par défaut pour les codes langue et les codes fonction 
            # lorsque ces informations sont absentes des données d’origine.

       

        
        """Attributs liés à publication et à l'impression
        Vu avec BmL : l'adresse bibliographique n'est pas systématiquement dans la même sous-zone, on en teste différentes"""
        ordre_adresse = [("210", "a"), ("210", "e"), ("214", "r")]
        self.lieu_edition = ""
        for subfield in ordre_adresse:
            if self.lieu_edition == "":
                self.lieu_edition = marc_record.get_fields(subfield[0])[0].get_subfields(subfield[1])[0] if len(marc_record.get_fields(subfield[0]))>0 and len(marc_record.get_fields(subfield[0])[0].get_subfields(subfield[1]))>0 else ""

       
        self.date_edition = "[s. d.]"
        self.editeur = "[s. n.]"
        self.lieu_impression = ""
        self.date_impression = ""

        if len(marc_record.get_fields("210"))>0 :
            # Valeurs par défaut "[s. d.]" / "[s. n.]" si la sous-zone est absente
            self.date_edition = marc_record.get_fields("210")[0].get_subfields("d")[0] if len(marc_record.get_fields("210")[0].get_subfields("d")) >0 else "[s. d.]"
            self.editeur = marc_record.get_fields("210")[0].get_subfields("c")[0] if len(marc_record.get_fields("210")[0].get_subfields("c"))>0 else "[s. n.]"
            self.lieu_impression = marc_record.get_fields("210")[0].get_subfields("e")[0] if len(marc_record.get_fields("210")[0].get_subfields("e"))>0 else ""
            self.date_impression = marc_record.get_fields("210")[0].get_subfields("g")[0] if len(marc_record.get_fields("210")[0].get_subfields("g"))>0 else ""

        
        """Date normalisée """
        self.date = marc_record.get_fields("100")[0].get_subfields("a")[0][9:17]if len(marc_record.get_fields("100"))>0 and len(marc_record.get_fields("100")[0].get_subfields("a"))>0   else ""
        regex_date = r"[0-9\.]{4}([0-9\. ]{4})?"
        search = re.fullmatch(regex_date, self.date)
        if search is None :
            self.date = re.sub(r"[^0-9\. ]", ".", self.date)
        # Un intervalle du type "1988...." indique une année connue, non la valeur "1"
        # VERIF QUE LE COMPORTEMENT SOIT BON, EN PARTICULIER CONCERNANT 
        # Les données manquantes doivent être remplacées par des points (par exemple 191.
        # cf mail de la semaine du 26
        if re.fullmatch(r"\d{4}\.{4}", self.date):
            self.date = self.date[:4]

        self.date = self.date.replace(" ", "")
        
        # Vérifier que la date n'est pas vide avant d'accéder au premier caractère ; sinon on risque une IndexError dans le Fichier5
        if self.date and self.date[0] == "'":
            self.date = self.date[1:]

        # Supprimer l'apostrophe éventuelle au début
        if self.date.startswith("'"):
            self.date = self.date[1:]

        # Compléter les années incomplètes par des points
        if re.fullmatch(r"[0-9.]{1,3}", self.date):
            self.date = self.date.ljust(4, ".")

        if len(self.date) == 8: 
            # ajouté le tiret du milieu
            self.date = insert (self.date, "-", 4)        

        
        """Code pays"""
        self.pays = ""
        if len(marc_record.get_fields("102"))>0 :
            self.pays = marc_record.get_fields("102")[0].get_subfields("a")[0] if len(marc_record.get_fields("102")[0].get_subfields("a"))>0 else ""
        
        # Conversion des anciens codes pays BML (DX, SP, XX, YU)
        code_pays = self.pays.strip().upper()
        self.pays = CODES_PAYS_BML.get(code_pays, code_pays.lower())
        if self.pays not in CODES_PAYS_VALIDES:
            self.pays = "xx"
        # mais mais mais
        if NOM_FICHIER == 'GMB_Fichier6':
            self.pays = "fr"
       
        """Zone de titre"""
        self.titre = marc_record.get_fields("200")[0].get_subfields("a")[0] if len(marc_record.get_fields("200"))>0 and len(marc_record.get_fields("200")[0].get_subfields("a"))>0   else ""
        self.complement_titre = marc_record.get_fields("200")[0].get_subfields("e")[0] if len(marc_record.get_fields("200"))>0 and len(marc_record.get_fields("200")[0].get_subfields("e"))>0   else ""
        self.titre_parallele = marc_record.get_fields("200")[0].get_subfields("d")[0] if len(marc_record.get_fields("200"))>0 and len(marc_record.get_fields("200")[0].get_subfields("d"))>0   else ""
        self.mention_resp_1 = marc_record.get_fields("200")[0].get_subfields("f")[0] if len(marc_record.get_fields("200"))>0 and len(marc_record.get_fields("200")[0].get_subfields("f"))>0   else ""
        # I02-02 / I02-03 : 200$g, première et deuxième occurrence
        liste_200g = marc_record.get_fields("200")[0].get_subfields("g") if len(marc_record.get_fields("200"))>0 else []
        self.mention_resp_2 = liste_200g[0] if len(liste_200g) > 0 else ""
        self.mention_resp_3 = liste_200g[1] if len(liste_200g) > 1 else ""

        """Informations techniques"""
        self.tirage = marc_record.get_fields("205")[0].get_subfields("a")[0] if len(marc_record.get_fields("205"))>0 and len(marc_record.get_fields("205")[0].get_subfields("a"))>0   else ""      
        self.importance_materielle = marc_record.get_fields("215")[0].get_subfields("a")[0] if len(marc_record.get_fields("215"))>0 and len(marc_record.get_fields("215")[0].get_subfields("a"))>0   else ""
        self.couleur = marc_record.get_fields("215")[0].get_subfields("c")[0] if len(marc_record.get_fields("215"))>0 and len(marc_record.get_fields("215")[0].get_subfields("c"))>0   else "" 
        self.dimensions = marc_record.get_fields("215")[0].get_subfields("d")[0] if len(marc_record.get_fields("215"))>0 and len(marc_record.get_fields("215")[0].get_subfields("d"))>0   else ""
        # on supprime la mention cm (sans oublier de faire une conversion mm -> cm au besoin)
        # ON FAIT EGALEMENT LE CHOIX DE MODIFIER LA MENTION "(épr.)"
        self.dimensions = self.dimensions.replace("(épr.)", "")
        if "mm" in self.dimensions:
            self.dimensions = self.dimensions.replace("mm", "")
            # conversion
            # e.g. 36 x 24 mm converti en 
            tmp = self.dimensions.split("x")
            if len(tmp) == 2:
                # print(type(tmp[0]), type(tmp[1]))
                dim1, dim2 = float(tmp[0]) / 10, float(tmp[1]) / 10
                self.dimensions = f"{dim1} x {dim2}"
            else:
                print("*"*50)
                print("PROBLEME")
                print(self.dimensions)
        self.dimensions = self.dimensions.replace("cm", "")

        # 200$b est souvent saisi entre crochets ([Photographie], [Manuscrit], etc.)
        categorie = marc_record.get_fields("200")[0].get_subfields("b")[0] if len(marc_record.get_fields("200"))>0 and len(marc_record.get_fields("200")[0].get_subfields("b"))>0 else ""
        categorie = categorie.strip()
        if categorie.startswith("[") and categorie.endswith("]"):
            categorie = categorie[1:-1].strip()
        self.categorie_technique = CATEGORIE_TECHNIQUE_NORMALISE.get(categorie.casefold(), "autre catégorie technique")

        self.coordo_geographiques = marc_record.get_fields("206")[0].get_subfields("d")[0] if len(marc_record.get_fields("206"))>0 and len(marc_record.get_fields("206")[0].get_subfields("d"))>0   else ""     
     
                   

        
        """Indexation : on a deux systèmes différents selon que le document fait partie de Photographes en RA ou non"""
        self.indexation = []
        

        photographes_ra = False

        z461 =  marc_record.get_fields("461")
        for field in z461 :
            if  "Photographes en Rhône-Alpes" in next(iter(field.get_subfields("t")), ""):
                photographes_ra = True


        if photographes_ra :
            zones_indexation = [("608", 1), ("600", 3), ("606", 3)]
            for zone in zones_indexation: 
                    fields = marc_record.get_fields(zone[0])
                    i = 0
                    while i < zone[1] and i <len(fields):
                        
                        str_field = field_to_string(fields[i])
                        self.indexation.append(marcfield2label(str_field))
                        i+= 1
            liste_617 = fields = marc_record.get_fields("617")[:2]
            i = 0
            while len(self.indexation)<15 and i <len(liste_617):
                
                str_field = field_to_string(liste_617[i])
                self.indexation.append(marcfield2label(str_field))
                i += 1
                
            
        else : 
            # LA PARTIE MARRANTE, A BIEN VERIFIER!!!! 
            zones_indexation = [("608", 1), ("600", 3), ("607", 2), ("606", 5), ("610", 1)]
            
            for zone in zones_indexation: 
                fields = marc_record.get_fields(zone[0])
                i = 0
                while i < zone[1] and i <len(fields):
                    str_field = field_to_string(fields[i])
                    self.indexation.append(marcfield2label(str_field))
                    i+= 1
    
      
            for zone in zones_indexation:
                fields = marc_record.get_fields(zone[0])
                i = 0
                while  i <len(fields) and len(self.indexation)<15: 
                    str_field = field_to_string(fields[i])
                    str_field = marcfield2label(str_field)
                    if str_field not in self.indexation :
                        self.indexation.append(str_field)                     
                    i += 1

        # Indexations arméniennes : sujets et lieux
        (
            self.indexation_armenienne_sujets,
            self.indexation_armenienne_lieux
        ) = extraire_indexations_armeniennes(marc_record)    
           
        self.notes = ""
        zones_notes = ["300", "305", "307", "321", "461", "488", "980"] #j'ai ajouté la zone 321 pour les notes bibliographiques

        # VERIF CE TRUC!!!
        # hmmm... CA M'EMBETE CE CODE!
        for zone in zones_notes:            
            fields = marc_record.get_fields(zone)                
            for field in fields:                    
                if zone == "300" :
                    note =  next(iter(field.get_subfields("a")), "")
                    if note != "":
                        self.notes += "Note générale : "
                        self.notes  += note if note.endswith(".") else note + ". "
                elif zone == "305":
                    note =  next(iter(field.get_subfields("a")), "")
                    if note != "":
                        self.notes += "Etats : "
                        self.notes  += note if note.endswith(".") else note + ". "
                elif zone == "307":
                    note =  next(iter(field.get_subfields("a")), "") 
                    if note != "":
                        self.notes += "Note sur les techniques : "
                        self.notes  += note if note.endswith(".") else note + ". "
                # J'ajoute la note bibliographique pour la zone 321 parce qu'il y a les sources bibliographiques pour les notices ARMENIENNES
                elif zone == "321":
                    for note in field.get_subfields("a"):
                        note = note.strip()
                        if note:
                            self.notes += "Note bibliographique : "
                            self.notes += note
                            if not note.endswith("."):
                                self.notes += "."
                            self.notes += " "
                elif zone == "461":
                    note =  next(iter(field.get_subfields("t")), "")
                    if note != "":
                        self.notes += "Fait partie d'une série : "
                        self.notes  += note if note.endswith(".") else note + ". " 
                elif zone == "488":
                    note =  next(iter(field.get_subfields("t")), "") + " : " + next(iter(field.get_subfields("i")), "")
                    
                    if note != " : ":
                        self.notes += "En lien avec le document : "
                        self.notes  += note if note.endswith(".") else note + ". " 
                elif zone == "980":
                    note =  next(iter(field.get_subfields("a")), "")
                    if note != "":
                        self.notes += "Note de contexte historique : "
                        self.notes  += note if note.endswith(".") else note + ". " 

     

               
        """Notes d'exemplaire"""     
        self.note_exemplaires = ""
        self.z316 = marc_record.get_fields("316")
        self.z317 = marc_record.get_fields("317")
        self.note_exemplaires = note_exe2string(self.z316, self.note_exemplaires)
        self.note_exemplaires = note_exe2string(self.z317, self.note_exemplaires)
        # A confirmer avec la BML : 317$a contient aussi le fonds (I72-02).
        # On conserve la règle initiale dans I73-01 tant que la ventilation n'est pas validée.
        
        """Cote et code-barre
        Les champs 930$a et 017$a sont présents dans le lot définitif, mais peuvent manquer
        dans certaines notices. On vérifie leur présence avant d'en récupérer les valeurs."""
        self.cote = ""
       
        # En cas de doublon 930$a, la cote de l'original est en général sans préfixe NUM_ SAUF documents nativement numériques ou la collection Vos cartes postales.
        
        # Première occurrence de 930$a
        cotes = [
            val.strip()
            for champ in marc_record.get_fields("930")
            for val in champ.get_subfields("a")
        ]

        premiere_cote = cotes[0] if cotes else ""

        # Document nativement numérique (215$a)
        numerique_natif = any(
            "1 photographie numérique" in " ".join(val.casefold().split())
            for champ in marc_record.get_fields("215")
            for val in champ.get_subfields("a")
        )

        # Collection Vos cartes postales (461$t)
        vos_cartes_postales = any(
            "vos cartes postales" in " ".join(val.casefold().split())
            for champ in marc_record.get_fields("461")
            for val in champ.get_subfields("t")
        )

        # Conservation ou suppression du préfixe NUM_
        if numerique_natif or vos_cartes_postales:
            self.cote = premiere_cote
        else:
            self.cote = re.sub(
                r"^NUM_", "", premiere_cote,
                flags=re.IGNORECASE
            )

        self.code_barre = marc_record.get_fields("017")[0].get_subfields("a")[0] if len(marc_record.get_fields("017"))>0 and len(marc_record.get_fields("017")[0].get_subfields("a"))>0 else ""
        # print(marc_record.get_fields("017")[0].get_subfields("a")[0])
        # print(self.code_barre)
        """Codes projets
        On initie la liste avec le code projet générique : maximum 3 colonnes"""
        codes = attribuer_codes_projet(marc_record)

        self.codes_projets = codes[0]
        self.code_projets2 = codes[1] if len(codes) > 1 else ""
        self.code_projets3 = codes[2] if len(codes) > 2 else ""

        self.liste_codes_projets = codes

        
        # Faisons juste une grosse disjonction de cas de fainéants...! 
        
    
    
            
    def get_subfield(self, field_tag, subfield_tag):
        #Récupère la valeur d'une sous-zone. Attention, ne fonctionne que si la zone et la sous-zone sont non répétables !
        record = self.marc_record
        field = record.get_fields(field_tag)[0] if record.get_fields(field_tag) else None
               
        res = ""
        if field is not None:
            
           
            res = field.get_subfields(subfield_tag)[0] if field.get_subfields(subfield_tag) else ""
        return res
        
    def __str__(self) :
        string = ""
        return string


## Création d'une liste de notices à partir du fichier ISO2709

In [15]:
list_indexation = []

In [16]:
list_records = []

## Modifications du 2 novembre

In [17]:
import re


## Retirer la ponctuation superflue au début d'une indexation.
def _nettoyer_indexation(valeur):
    valeur = re.sub(r"^[\s\-–—.,;:]+", "", valeur)
    valeur = re.sub(r"\s+([,;:])", r"\1", valeur)
    return re.sub(r" {2,}", " ", valeur).strip()


## Harmonisation des noms de personnes et des niveaux géographiques.
def formater_indexation(champ, field_to_string, marcfield2label):
    if champ.tag == "600":
        valeur = ""
        for sous_zone in champ.subfields:
            texte = sous_zone.value.strip()
            if not texte:
                continue
            if sous_zone.code == "a":
                valeur += (" ; " if valeur else "") + texte
            elif sous_zone.code == "b":
                valeur += (", " if valeur else "") + texte
            elif sous_zone.code == "c":
                valeur += (" ; " if valeur else "") + texte
            elif sous_zone.code == "f":
                valeur += f" ({texte})"
            else:
                valeur += f" ; ${sous_zone.code} {texte}"
        return _nettoyer_indexation(valeur)

    if champ.tag == "617":
        valeur = ""
        separateurs = {
            "a": ", ",
            "b": ", ",
            "c": "; ",
            "d": "; ",
            "e": "; ",
            "k": "; ",
        }
        for sous_zone in champ.subfields:
            texte = sous_zone.value.strip()
            if not texte:
                continue
            if not valeur:
                valeur = texte
            elif sous_zone.code in separateurs:
                valeur += separateurs[sous_zone.code] + texte
            else:
                valeur += f"; ${sous_zone.code} {texte}"
        return _nettoyer_indexation(valeur)

    return _nettoyer_indexation(marcfield2label(field_to_string(champ)))


## MODIFICATION : Assemble les notes avec un espace entre les phrases.
def _normaliser_notes(marc_record):
    notes = []

    def ajouter(prefixe, texte):
        texte = texte.strip()
        if texte:
            note = prefixe + texte
            notes.append(note if note.endswith(".") else note + ".")

    for champ in marc_record.get_fields("300"):
        ajouter("Note générale : ", next(iter(champ.get_subfields("a")), ""))
    for champ in marc_record.get_fields("305"):
        ajouter("Etats : ", next(iter(champ.get_subfields("a")), ""))
    for champ in marc_record.get_fields("307"):
        ajouter("Note sur les techniques : ", next(iter(champ.get_subfields("a")), ""))
    for champ in marc_record.get_fields("321"):
        for texte in champ.get_subfields("a"):
            ajouter("Note bibliographique : ", texte)
    for champ in marc_record.get_fields("461"):
        ajouter("Fait partie d'une série : ", next(iter(champ.get_subfields("t")), ""))
    for champ in marc_record.get_fields("488"):
        elements = [
            next(iter(champ.get_subfields(code)), "").strip()
            for code in ("t", "i")
        ]
        ajouter(
            "En lien avec le document : ",
            " : ".join(element for element in elements if element),
        )
    for champ in marc_record.get_fields("980"):
        ajouter("Note de contexte historique : ", next(iter(champ.get_subfields("a")), ""))

    return " ".join(notes)


## MODIFICATION : Sépare les notes successives d'exemplaire.
def _normaliser_notes_exemplaire(marc_record):
    notes = [
        valeur.strip()
        for zone in ("316", "317")
        for champ in marc_record.get_fields(zone)
        for valeur in champ.get_subfields("a")
        if valeur.strip()
    ]
    return " ".join(
        valeur if valeur.endswith(".") else valeur + "."
        for valeur in notes
    )


## MODIFICATION : Ajoute les mentions bibliographiques manquantes entre crochets.
def normaliser_mention_bibliographique(valeur, abreviation, valeur_defaut):
    valeur = str(valeur or "").strip()
    if not valeur:
        return valeur_defaut

    valeur_sans_crochets = valeur.removeprefix("[").removesuffix("]").strip()
    if re.sub(r"\s+", " ", valeur_sans_crochets).casefold() == abreviation:
        return valeur_defaut
    return valeur


def appliquer_corrections_bnf(
    notice, marc_record, nom_fichier, field_to_string, marcfield2label
):
    ## MODIFICATION : Les titres alternatifs viennent de toutes les zones 540$a.
    notice.titre_alternatif = " ; ".join(
        valeur.strip()
        for champ in marc_record.get_fields("540")
        for valeur in champ.get_subfields("a")
        if valeur.strip()
    )

    ## MODIFICATION : LES FICHIERS 1 ET 6 RECOIVENT LEUR VALEUR MATÉRIELLE OBLIGATOIRE.
    if not notice.importance_materielle.strip():
        if nom_fichier == "GMB_Fichier1":
            notice.importance_materielle = "1 carte postale"
        elif nom_fichier == "GMB_Fichier6":
            notice.importance_materielle = "1 photographie"

    ## MODIFICATION : Normalise lieu, éditeur et date de publication.
    notice.lieu_edition = normaliser_mention_bibliographique(
        notice.lieu_edition, "s. l.", "[s. l.]"
    )
    notice.editeur = normaliser_mention_bibliographique(
        notice.editeur, "s. n.", "[s. n.]"
    )
    notice.date_edition = normaliser_mention_bibliographique(
        notice.date_edition, "s. d.", "[s. d.]"
    )

    ## MODIFICATION : La couleur est normalisée à partir des mentions en 215$c.
    mentions_couleur = [
        valeur.casefold()
        for champ in marc_record.get_fields("215")
        for valeur in champ.get_subfields("c")
    ]
    couleur_explicite = any(
        re.search(r"\b(?:couleur|color(?:é|ée|és|ées)|colori(?:é|ée|és|ées)|coloris(?:é|ée|és|ées))\b", valeur)
        for valeur in mentions_couleur
    )
    noir_et_blanc_explicite = any(
        re.search(r"\bnoir(?:e)?s?\s+et\s+blanc(?:he)?s?\b", valeur)
        for valeur in mentions_couleur
    )
    if couleur_explicite != noir_et_blanc_explicite:
        notice.couleur = "couleur" if couleur_explicite else "noir et blanc"
    else:
        notice.couleur = ""

    ## MODIFICATION : Les noms de personnes et les tirets superflus sont harmonisés.
    photographes_ra = any(
        "Photographes en Rhône-Alpes" in next(iter(champ.get_subfields("t")), "")
        for champ in marc_record.get_fields("461")
    )
    notice.indexation = []
    if photographes_ra:
        zones_indexation = [("608", 1), ("600", 3), ("606", 3)]
        for zone, limite in zones_indexation:
            notice.indexation.extend(
                formater_indexation(champ, field_to_string, marcfield2label)
                for champ in marc_record.get_fields(zone)[:limite]
            )
    else:
        zones_indexation = [("608", 1), ("600", 3), ("607", 2), ("606", 5), ("610", 1)]
        for zone, limite in zones_indexation:
            notice.indexation.extend(
                formater_indexation(champ, field_to_string, marcfield2label)
                for champ in marc_record.get_fields(zone)[:limite]
            )
        for zone, _ in zones_indexation:
            for champ in marc_record.get_fields(zone):
                valeur = formater_indexation(champ, field_to_string, marcfield2label)
                if len(notice.indexation) >= 15:
                    break
                if valeur and valeur not in notice.indexation:
                    notice.indexation.append(valeur)

    ## MODIFICATION : Chaque occurrence 617 garde sa graphie et ses sous-zones.
    notice.indexation.extend(
        valeur
        for champ in marc_record.get_fields("617")
        if (
            valeur := formater_indexation(
                champ, field_to_string, marcfield2label
            )
        )
    )

    ## MODIFICATION : Les notes bibliographiques sont séparées par une espace.
    notice.notes = _normaliser_notes(marc_record)

    ## MODIFICATION : Les notes d'exemplaire sont séparées après chaque point.
    notice.note_exemplaires = _normaliser_notes_exemplaire(marc_record)


In [18]:
#Changer ici le chemin du fichier avec le fichier définitif

# CORRECTIF ENCODAGE (ajouté) : les fichiers GMB sont en UTF-8, mais leur label (position 9) est vide,
# donc MARCReader(force_utf8=False) les décode en MARC-8 et produit des caractères erronés (é -> ©♭).
# Avec force_utf8=True, toutes les notices sont rejetées (None) à cause d'octets parasites non UTF-8
# (ex. \x03\xa2 en fin d'URL). On lit donc les notices une à une, on neutralise ces octets
# (remplacés par des espaces, sans changer la longueur) et on force le décodage UTF-8.
from pymarc import Record

def lire_records_utf8(fh):
    for chunk in fh.read().split(b'\x1d'):  # \x1d = fin de notice ISO 2709
        if len(chunk) < 24:
            continue
        buf = bytearray(chunk)
        while True:
            try:
                bytes(buf).decode('utf-8')
                break
            except UnicodeDecodeError as e:
                for k in range(e.start, e.end):
                    buf[k] = 0x20
        yield Record(data=bytes(buf) + b'\x1d', to_unicode=True, force_utf8=True)

# AJOUT : nom du fichier à convertir, réutilisé pour nommer le fichier Excel final
# (ex. 'GMB_Fichier2' -> GMB_Fichier2_partie_1.xlsx). Changer uniquement ici.
NOM_FICHIER = 'GMB_Fichier8'


# idx = 0

with open(NOM_FICHIER, 'rb') as fh:  # MODIF : 'GMB_Fichier2' remplacé par NOM_FICHIER
    reader = lire_records_utf8(fh)  # CORRECTIF : remplace MARCReader(fh, force_utf8=False)
    for record in tqdm(reader,desc="Processing Records"):
        notice = notice_unimarc(record)
        ## MODIFICATION : Applique les règles BnF avant la création de la matrice.
        appliquer_corrections_bnf(notice, record, NOM_FICHIER, field_to_string, marcfield2label)

        list_records.append(notice)

        list_indexation.extend(notice.indexation)   
        #idx += 1
        #if idx > 150: break
         


Processing Records: 0it [00:00, ?it/s]

Processing Records: 55it [00:00, 289.22it/s]c:\Users\mathi\OneDrive\Bureau\Projet Numelyo\Projet Numelyo\SRUextractionMODIFIE.py:162: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/srw:diagnostics'
  if (self.result[0].find("//srw:diagnostics",
c:\Users\mathi\OneDrive\Bureau\Projet Numelyo\Projet Numelyo\SRUextractionMODIFIE.py:177: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/srw:numberOfRecords'
  if (self.result[0].find("//srw:numberOfRecords",
c:\Users\mathi\OneDrive\Bureau\Projet Numelyo\Projet Numelyo\SRUextractionMODIFIE.py:180: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/srw:numberOfRecords'
  self.nb_results = int(self.result[0].find("//sr

In [19]:
2130 + 517 + 1315 + 14470 + 12349 + 11562 + 8487 + 9934 + 67246 + 28

128038

In [20]:
print(len(list_records))

9934


In [21]:
counter = Counter(list_indexation)
subjects = [(count, subject) for subject, count in counter.items()]
df_subjects = pandas.DataFrame(subjects, columns = ["Count", "Subjects"])
df_subjects.to_excel("./test_indexation.xlsx")

## Création de la matrice

### Mapping
Concordance des informations entre les colonnes de la matrice et les attributs de l'objet notice unimarc

In [22]:
record2matrice = {
    "I01-01": "titre",
    "I01-02" : "complement_titre",
    "I01-03" : "titre_parallele",
    "I02-01" : "mention_resp_1",
    "I02-02" : "mention_resp_2",
    "I02-03" : "mention_resp_3",
    "I12-01" : "langue1", 
    "I12-02" : "langue2", 
    "I12-03" : "langue3",
    "I21-01" : "Auteur1.nna",
    "I21-02" : "Auteur1.nom",
    "I21-03" : "Auteur1.prenom",
    "I21-04" : "Auteur1.fonction",
    "I21-05" : "Auteur1.type",
    "I22-01" : "Auteur2.nna",
    "I22-02" : "Auteur2.nom",
    "I22-03" : "Auteur2.prenom",
    "I22-04": "Auteur2.fonction",
    "I22-05": "Auteur2.type",
    "I23-01" : "Auteur3.nna",
    "I23-02" : "Auteur3.nom",
    "I23-03" : "Auteur3.prenom",
    "I23-04": "Auteur3.fonction",
    "I23-05": "Auteur3.type",
    "I31-02" : "lieu_edition",
    "I32-01" : "date_edition",
    "I31-03" : "editeur",
    "I33-01" : "lieu_impression",
    "I33-03" : "date_impression",
    "I32-03" : "date",
    "I41-01" : "tirage",
    "I42-01": "importance_materielle",
    "I42-03" : "couleur", 
    "I42-04" : "dimensions",
    "I43-03" : "categorie_technique",
    "I46-02" : "coordo_geographiques",
    "I51-01" : "notes",
    "I11-01" : "pays",
    "I71-01" : "codes_projets",   
    "I71-01": "codes_projets",
    # Attribuions les deux codes complémentaires
    "I71-02": "code_projets2",
    "I71-03": "code_projets3",
    # I72-02 : nom du fonds (317$a), à confirmer avec la BML
    "I72-01" : "provenance",
    "I72-03" : "cote",
    "I72-04" : "ark",
    "I72-05" : "code_barre",
    "I73-01" : "note_exemplaires"
   
}


### Liste des colonnes

In [23]:
COLS_MATRICE = ["I01-01", "I01-02", "I01-03", "I01-04", "I02-01", 
                "I02-02", "I02-03", "I11-01", "I12-01", "I12-02", 
                "I12-03", "I21-01", "I21-02", "I21-03", "I21-04", 
                "I21-05", "I22-01", "I22-02", "I22-03", "I22-04", 
                "I22-05", "I23-01", "I23-02", "I23-03", "I23-04", 
                "I23-05", "I31-02", "I31-03", "I32-01", "I32-03", 
                "I33-01", "I33-02", "I33-03", "I34-01", "I41-01", 
                "I42-01", "I42-03", "I42-04", "I43-01", "I43-02", 
                "I43-03", "I44-01", "I45-01", "I46-02", "I51-01", 
                "I52-01", "I71-01", "I71-02", "I71-03", "I72-01", 
                "I72-02", "I72-03", "I72-04", "I72-05", "I73-01", 
                "I73-02", "I73-03"]

len(COLS_MATRICE)


57

### Transformation des notices en matrice

In [24]:
def col2attr(key, record, dic) : 
    """Fonction qui permet d'aller récupérer les attributs à partir du dictionnaire ci-dessous"""
    if "." in dic[key] :
        el = dic[key].split(".")
        auteur = getattr(record, el[0])
        res = getattr(auteur, el[1])
    else : 
        res = getattr(record, dic[key])
    return res

In [25]:
## MODIFICATION : La colonne I52 reprend les titres alternatifs en 540$a.
record2matrice["I52-01"] = "titre_alternatif"

records_data = []
for record in tqdm(list_records, desc="Processing Records"):
    d = {"id": record.num_notice}

    for col in COLS_MATRICE:
        if col in record2matrice:
            d[col] = col2attr(col, record, record2matrice)
        else:
            d[col] = ""

    # Indexations en colonnes supplémentaires : la nomenclature I61-00-XX
    # est conservée du script initial, à vérifier avec le format du livrable final.
    ## MODIFICATION : Le créneau 00 reste inchangé ; les genres ne sont pas remappés.
    for i, item in enumerate(record.indexation):
        res = item.replace("-- frTICO", "").replace("-- rameau", "")
        # Supprimer les suffixes « -- frTico » et « -- rameau » ajoutés à la suite des sujets. 
        res = res.strip()
        # Supprimer uniquement les espaces au début et à la fin ; sinon nous enlevons un caractère à la fin !
        d[f"I61-00-{i+1:02d}"] = res

    # UPDATE!!! Codes projet supplémentaires à partir de I71-04. ON AJOUTE LES CODES SUPPLÉMENTAIRES DANS LA MATRICE
    for position, code in enumerate(
        record.liste_codes_projets[3:], start=4
    ):
        d[f"I71-{position:02d}"] = code

    
    # Indexations arméniennes : sujets
    for i, valeur in enumerate(
        record.indexation_armenienne_sujets, start=1
    ):
        d[f"I61-01-{i:02d}"] = valeur

    # Indexations arméniennes : lieux
    for i, valeur in enumerate(
        record.indexation_armenienne_lieux, start=1
    ):
        d[f"I63-01-{i:02d}"] = valeur

    records_data.append(d)

# Vérifier que toutes les correspondances sont présentes dans la matrice attendue
colonnes_non_declarees = set(record2matrice) - set(COLS_MATRICE)
if colonnes_non_declarees:
    raise ValueError(f"Colonnes absentes de COLS_MATRICE : {sorted(colonnes_non_declarees)}")

matrice = pandas.DataFrame(records_data)


Processing Records:   0%|          | 0/9934 [00:00<?, ?it/s]

Processing Records: 100%|██████████| 9934/9934 [00:00<00:00, 12321.48it/s]


In [26]:
with pandas.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    print(matrice.head(3))


           id             I01-01 I01-02 I01-03 I01-04  \
0  0003368326   A. L. de Jussieu                        
1  0003368324  A. L. de Jussieu.                        
2  0003373737               1793                        

                                     I02-01  \
0            [Tardieu, Ambroise, 1788-1841]   
1                 [Féart, Adrien 1813-1880]   
2  [Raffet, Denis-Auguste-Marie, 1804-1860]   

                                     I02-02 I02-03 I11-01 I12-01 I12-02  \
0                                                      xx                 
1                 [Féart, Adrien 1813-1880]            xx                 
2  [Raffet, Denis-Auguste-Marie, 1804-1860]            xx                 

  I12-03    I21-01   I21-02            I21-03 I21-04    I21-05    I22-01  \
0                   Tardieu          Ambroise   0410  Personne  10615176   
1         14973720    Féart  Adrien 1813-1880   0410  Personne  14973720   
2         13012888   Raffet           Auguste   0990 

In [27]:
import io
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from reportlab.lib.pagesizes import A4
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak, HRFlowable
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.pdfgen import canvas

# ==============================================================================
# 1. CANVAS AVEC NORMES INSTITUTIONNELLES ET NUMÉROTATION DYNAMIQUE
# ==============================================================================
class NumberedCanvas(canvas.Canvas):
    """
    Canvas avec gestion dynamique des numéros de page "Page X sur Y",
    en-tête de document et pied de page institutionnel.
    """
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._saved_page_states = []

    def showPage(self):
        self._saved_page_states.append(dict(self.__dict__))
        self._startPage()

    def save(self):
        num_pages = len(self._saved_page_states)
        for state in self._saved_page_states:
            self.__dict__.update(state)
            self.draw_page_decorations(num_pages)
            super().showPage()
        super().save()

    def draw_page_decorations(self, page_count):
        self.saveState()
        self.setFont("Helvetica", 8)
        self.setFillColor(colors.HexColor("#666666"))
        
        # En-tête (page 2+)
        if self._pageNumber > 1:
            self.drawString(54, 800, "Rapport d'Analyse Exploratoire et Profilage de Dataframe")
            self.setStrokeColor(colors.HexColor("#dcdcdc"))
            self.setLineWidth(0.5)
            self.line(54, 792, 558, 792)
            
        # Pied de page (Toutes pages)
        text = f"Page {self._pageNumber} sur {page_count}"
        self.drawRightString(558, 36, text)
        self.drawString(54, 36, "RAPPORT AUTOMATISÉ — AUDIT DE QUALITÉ DES DONNÉES")
        self.setStrokeColor(colors.HexColor("#dcdcdc"))
        self.setLineWidth(0.5)
        self.line(54, 48, 558, 48)
        
        self.restoreState()


# ==============================================================================
# 2. GÉNÉRATION DES GRAPHIQUES D'ANALYSE EXPLORATOIRE
# ==============================================================================
def generer_graphiques_dataframe(df):
    """
    Génère 3 graphiques analytiques haute définition basés uniquement sur les métriques du DataFrame.
    """
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    
    # --- Graphique 1 : Distribution de la complétude par tranche (%) ---
    compl_rates = (df.notna().sum() / len(df)) * 100
    bins = [0, 0.001, 25, 50, 75, 99.999, 100]
    labels_bins = ['0% (Vide)', '1-25%', '26-50%', '51-75%', '76-99%', '100% (Complet)']
    tranches = pd.cut(compl_rates, bins=bins, labels=labels_bins, include_lowest=True).value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(7.5, 3.2), dpi=300)
    colors_bars = ['#d9534f', '#f0ad4e', '#f7d794', '#74b9ff', '#0984e3', '#2b5c8f']
    bars = ax.bar(tranches.index.astype(str), tranches.values, color=colors_bars, width=0.55, edgecolor='none')
    ax.set_ylabel('Nombre de Colonnes', fontsize=9.5, fontweight='bold', color='#333333')
    ax.set_title('Distribution de la complétude à travers toutes les colonnes', fontsize=11, fontweight='bold', pad=12, color='#1b365d')
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, f'{int(h)}', ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#2b5c8f')
    ax.set_ylim(0, max(tranches.values.max() + 3, 5))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    chart1_path = 'temp_chart_distribution.png'
    fig.savefig(chart1_path, bbox_inches='tight', dpi=300)
    plt.close()

    # --- Graphique 2 : Profil des types de données Pandas ---
    dtypes_counts = df.dtypes.astype(str).value_counts()
    fig, ax = plt.subplots(figsize=(5.5, 2.8), dpi=300)
    wedges, texts, autotexts = ax.pie(
        dtypes_counts.values, 
        labels=dtypes_counts.index, 
        autopct='%1.1f%%',
        startangle=140,
        colors=['#2b5c8f', '#6baed6', '#9ecae1', '#c6dbef'],
        textprops=dict(fontsize=9, fontweight='bold')
    )
    for autotext in autotexts:
        autotext.set_color('white')
    ax.set_title('Répartition des types de données (dtypes)', fontsize=10.5, fontweight='bold', pad=10, color='#1b365d')
    plt.tight_layout()
    chart2_path = 'temp_chart_dtypes.png'
    fig.savefig(chart2_path, bbox_inches='tight', dpi=300)
    plt.close()

    # --- Graphique 3 : Top 15 des colonnes les plus denses vs les plus creuses ---
    top_dense = compl_rates.nlargest(15).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(7.5, 3.8), dpi=300)
    bars_dense = ax.barh(top_dense.index, top_dense.values, color='#2b5c8f', height=0.6)
    ax.set_xlim(0, 110)
    ax.set_xlabel('Taux de complétude (%)', fontsize=9, fontweight='bold')
    ax.set_title('Top 15 des colonnes les plus denses (% de remplissage)', fontsize=10.5, fontweight='bold', pad=10, color='#1b365d')
    for bar in bars_dense:
        w = bar.get_width()
        ax.text(w + 1, bar.get_y() + bar.get_height()/2, f'{w:.1f}%', va='center', ha='left', fontsize=8, fontweight='bold', color='#2b5c8f')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    chart3_path = 'temp_chart_top15.png'
    fig.savefig(chart3_path, bbox_inches='tight', dpi=300)
    plt.close()

    return chart1_path, chart2_path, chart3_path


# ==============================================================================
# 3. GÉNÉRATION AUTOMATIQUE DU RAPPORT PDF SUR TOUTES LES COLONNES
# ==============================================================================
def generer_pdf_profilage_dataframe(df, output_pdf_path='rapport_profilage_dataframe.pdf'):
    """
    Parcourt l'intégralité des colonnes du DataFrame et produit le rapport PDF complet.
    """
    chart1_path, chart2_path, chart3_path = generer_graphiques_dataframe(df)

    doc = SimpleDocTemplate(
        output_pdf_path,
        pagesize=A4,
        leftMargin=54,
        rightMargin=54,
        topMargin=54,
        bottomMargin=54
    )
    compl_rates = (df.notna().sum() / len(df)) * 100
    styles = getSampleStyleSheet()
    primary_color = colors.HexColor("#1b365d")
    secondary_color = colors.HexColor("#2b5c8f")
    text_color = colors.HexColor("#222222")

    styles['Normal'].textColor = text_color
    styles['Normal'].fontSize = 9.5
    styles['Normal'].leading = 13

    # Style definitions
    title_style = ParagraphStyle('DocTitle', parent=styles['Normal'], fontName='Helvetica-Bold', fontSize=19, leading=23, textColor=primary_color, spaceAfter=4)
    subtitle_style = ParagraphStyle('DocSubTitle', parent=styles['Normal'], fontName='Helvetica', fontSize=10.5, leading=14, textColor=secondary_color, spaceAfter=12)
    h1_style = ParagraphStyle('H1', parent=styles['Normal'], fontName='Helvetica-Bold', fontSize=12.5, leading=16, textColor=primary_color, spaceBefore=12, spaceAfter=8, keepWithNext=True)
    body_style = ParagraphStyle('BodyCustom', parent=styles['Normal'], spaceAfter=8)
    callout_style = ParagraphStyle('Callout', parent=styles['Normal'], fontSize=8.8, leading=12.5, textColor=colors.HexColor("#1b365d"))
    cell_style = ParagraphStyle('Cell', parent=styles['Normal'], fontSize=7.5, leading=10)
    cell_bold = ParagraphStyle('CellBold', parent=styles['Normal'], fontName='Helvetica-Bold', fontSize=8, leading=10, textColor=colors.white)

    story = []

    # --------------------------------------------------------------------------
    # EN-TÊTE DU DOCUMENT
    # --------------------------------------------------------------------------
    story.append(Paragraph("Rapport Automatisé d'Analyse et Profilage de DataFrame", title_style))
    story.append(Paragraph(f"Diagnostic exhaustif de la structure et de la qualité des données (N = {len(df)} lignes, {len(df.columns)} colonnes)", subtitle_style))
    story.append(HRFlowable(width="100%", thickness=1.5, color=primary_color, spaceAfter=10))

    # Métriques générales
    n_rows, n_cols = df.shape
    total_cells = df.size
    filled_cells = df.notna().sum().sum()
    null_cells = df.isna().sum().sum()
    global_fill_rate = (filled_cells / total_cells) * 100 if total_cells > 0 else 0
    full_cols_count = (df.notna().sum() == n_rows).sum()
    empty_cols_count = (df.notna().sum() == 0).sum()
    partial_cols_count = n_cols - full_cols_count - empty_cols_count

    # Résumé exécutif
    summary_html = f"""
    <b>SYNTHÈSE DU DATAFRAME :</b><br/>
    Le jeu de données analysé comporte <b>{n_rows} lignes</b> et <b>{n_cols} colonnes</b> (soit une matrice de <b>{total_cells} cellules</b>).<br/>
    • <b>Complétude globale :</b> <b>{global_fill_rate:.2f}%</b> ({filled_cells} cellules renseignées, {null_cells} cellules manquantes/NaN).<br/>
    • <b>Répartition des colonnes :</b> <b>{full_cols_count}</b> colonnes complètes (100%), <b>{partial_cols_count}</b> colonnes partiellement remplies, et <b>{empty_cols_count}</b> colonnes entièrement vides (0%).
    """
    summary_table = Table([[Paragraph(summary_html, callout_style)]], colWidths=[486])
    summary_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor("#f0f4f8")),
        ('BOX', (0,0), (-1,-1), 1, colors.HexColor("#c2d4e5")),
        ('PADDING', (0,0), (-1,-1), 7),
    ]))
    story.append(summary_table)
    story.append(Spacer(1, 10))

    # --------------------------------------------------------------------------
    # SECTION 1 : VUE D'ENSEMBLE STRUCTURELLE (TABLEAU KPIS)
    # --------------------------------------------------------------------------
    story.append(Paragraph("1. Indicateurs Globaux du Dataframe", h1_style))
    
    kpi_data = [
        [Paragraph("Métrique Global", cell_bold), Paragraph("Valeur", cell_bold), Paragraph("Interprétation & Impact", cell_bold)],
        [Paragraph("Nombre de Lignes (N)", cell_style), Paragraph(f"{n_rows:,}", cell_style), Paragraph("Volume total d'enregistrements/notices", cell_style)],
        [Paragraph("Nombre de Colonnes (P)", cell_style), Paragraph(f"{n_cols:,}", cell_style), Paragraph("Dimensionnalité de l'espace de variables", cell_style)],
        [Paragraph("Taux de Remplissage Global", cell_style), Paragraph(f"{global_fill_rate:.2f} %", cell_style), Paragraph("Densité globale de l'information stockée", cell_style)],
        [Paragraph("Colonnes 100% Complètes", cell_style), Paragraph(f"{full_cols_count} ({full_cols_count/n_cols*100:.1f}%)", cell_style), Paragraph("Variables à haute fiabilité (sans manquants)", cell_style)],
        [Paragraph("Colonnes Partiellement Remplies", cell_style), Paragraph(f"{partial_cols_count} ({partial_cols_count/n_cols*100:.1f}%)", cell_style), Paragraph("Variables facultatives ou conditionnelles", cell_style)],
        [Paragraph("Colonnes 100% Vides (NaN)", cell_style), Paragraph(f"{empty_cols_count} ({empty_cols_count/n_cols*100:.1f}%)", cell_style), Paragraph("Candidates à la suppression/nettoyage", cell_style)],
        [Paragraph("Mémoire Estimée (RAM)", cell_style), Paragraph(f"{df.memory_usage(deep=True).sum() / 1024:.1f} KB", cell_style), Paragraph("Empreinte mémoire du Dataframe", cell_style)]
    ]
    
    t_kpis = Table(kpi_data, colWidths=[135, 95, 256])
    t_kpis.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), primary_color),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#dcdcdc")),
        ('PADDING', (0,0), (-1,-1), 4.5),
        ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor("#f9f9f9")]),
    ]))
    story.append(t_kpis)
    story.append(Spacer(1, 10))

    # --------------------------------------------------------------------------
    # SECTION 2 : VISUALISATIONS GRAPHIQUES
    # --------------------------------------------------------------------------
    story.append(Paragraph("2. Distribution et Répartition des Données", h1_style))
    
    # Grid 2 images cote a cote / superposees
    story.append(Image(chart1_path, width=486, height=207))
    story.append(Spacer(1, 8))
    
    img_grid_data = [
        [Image(chart2_path, width=235, height=120), Image(chart3_path, width=245, height=124)]
    ]
    t_img_grid = Table(img_grid_data, colWidths=[240, 246])
    t_img_grid.setStyle(TableStyle([
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('LEFTPADDING', (0,0), (-1,-1), 0),
        ('RIGHTPADDING', (0,0), (-1,-1), 0),
        ('BOTTOMPADDING', (0,0), (-1,-1), 0),
        ('TOPPADDING', (0,0), (-1,-1), 0),
    ]))
    story.append(t_img_grid)
    story.append(Spacer(1, 12))

    # --------------------------------------------------------------------------
    # SECTION 3 : ANALYSE EXHAUSTIVE DE TOUTES LES COLONNES (POURTOUT DF)
    # --------------------------------------------------------------------------
    story.append(PageBreak())
    story.append(Paragraph(f"3. Dictionnaire Statistique des {n_cols} Colonnes", h1_style))
    story.append(Paragraph("Audit complet de chaque variable : type de donnée, valeurs non-nulles, taux de complétude, cardinalité (valeurs uniques) et exemples de valeurs représentatives.", body_style))

    # En-tête du tableau exhaustif
    all_cols_table_data = [[
        Paragraph("#", cell_bold),
        Paragraph("Nom Colonne", cell_bold),
        Paragraph("Type", cell_bold),
        Paragraph("Complétude (%)", cell_bold),
        Paragraph("Val. Uniques", cell_bold),
        Paragraph("Exemples / Valeurs observées", cell_bold)
    ]]

    for i, col_name in enumerate(df.columns, start=1):
        s = df[col_name]
        cnt_val = s.notna().sum()
        pct_comp = (cnt_val / n_rows) * 100
        n_unique = s.nunique(dropna=True)
        dtype_str = str(s.dtype)
        
        # Extraire des exemples pertinents
        non_null_samples = s.dropna().unique().tolist()
        if non_null_samples:
            sample_str = ", ".join([str(x) for x in non_null_samples[:3]])
            if len(sample_str) > 38:
                sample_str = sample_str[:35] + "..."
        else:
            sample_str = "<i>(100% NaN)</i>"

        # Badge de couleur pour la complétude
        if pct_comp == 100:
            comp_formatted = f"<b>{pct_comp:.0f}%</b> ({cnt_val}/{n_rows})"
        elif pct_comp == 0:
            comp_formatted = f"<font color='#d9534f'><b>0%</b> (0/{n_rows})</font>"
        else:
            comp_formatted = f"{pct_comp:.1f}% ({cnt_val}/{n_rows})"

        all_cols_table_data.append([
            Paragraph(str(i), cell_style),
            Paragraph(f"<b>{col_name}</b>", cell_style),
            Paragraph(dtype_str, cell_style),
            Paragraph(comp_formatted, cell_style),
            Paragraph(str(n_unique), cell_style),
            Paragraph(sample_str, cell_style)
        ])

    # Découpage du tableau pour éviter des ruptures de pages désordonnées
    t_all_cols = Table(all_cols_table_data, colWidths=[20, 85, 45, 80, 50, 206], repeatRows=1)
    t_all_cols.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), primary_color),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#e0e0e0")),
        ('PADDING', (0,0), (-1,-1), 3.5),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor("#f9f9f9")]),
    ]))
    story.append(t_all_cols)
    story.append(Spacer(1, 14))

    # --------------------------------------------------------------------------
    # SECTION 4 : SYNTHÈSE QUALITÉ ET RECOMMANDATIONS AUTOMATISÉES
    # --------------------------------------------------------------------------
    story.append(Paragraph("4. Synthèse Qualité et Recommandations de Nettoyage", h1_style))

    # Construction dynamique des recommandations
    recos = []
    if empty_cols_count > 0:
        empty_names = df.columns[df.notna().sum() == 0].tolist()
        empty_str = ", ".join([f"<code>{c}</code>" for c in empty_names[:5]])
        if len(empty_names) > 5:
            empty_str += f" (+{len(empty_names)-5} autres)"
        recos.append(f"• <b>Suppression des colonnes inactives ({empty_cols_count}) :</b> Les colonnes {empty_str} ne contiennent aucune donnée. Leur suppression permettra d'alléger l'empreinte mémoire et de simplifier le schéma.")

    low_comp_cols = df.columns[(compl_rates > 0) & (compl_rates < 50)].tolist()
    if low_comp_cols:
        low_str = ", ".join([f"<code>{c}</code>" for c in low_comp_cols[:4]])
        recos.append(f"• <b>Vérification des colonnes à faible densité ({len(low_comp_cols)}) :</b> Les champs {low_str} possèdent un taux d'incomplétude élevé (>50%). Assurez-vous que leur caractère facultatif est bien conforme à votre référentiel de données.")

    high_card_cols = [c for c in df.select_dtypes(include=['object']).columns if df[c].nunique() == n_rows and n_rows > 1]
    if high_card_cols:
        recos.append(f"• <b>Identifiants uniques détectés :</b> Les colonnes {', '.join([f'<code>{c}</code>' for c in high_card_cols])} disposent d'une cardinalité égale au nombre de lignes. Elles peuvent servir de clés primaires.")

    if not recos:
        recos.append("• <b>Excellente intégrité des données :</b> Aucune anomalie majeure de structure ou de complétude n'a été détectée.")

    reco_html = "<br/><br/>".join(recos)
    reco_table = Table([[Paragraph(reco_html, body_style)]], colWidths=[486])
    reco_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor("#fcf8e3")),
        ('BOX', (0,0), (-1,-1), 1, colors.HexColor("#fbeed5")),
        ('PADDING', (0,0), (-1,-1), 8),
    ]))
    story.append(reco_table)

    # Génération du document final
    doc.build(story, canvasmaker=NumberedCanvas)
    
    # Suppression des images temporaires
    for p in [chart1_path, chart2_path, chart3_path]:
        if os.path.exists(p):
            os.remove(p)

    print(f"✅ Rapport PDF généré avec succès : {output_pdf_path}")


# ==============================================================================
# 5. TEST AUTO-CONTENUS ET EXÉCUTION
# ==============================================================================
if __name__ == '__main__':
    
    df = matrice
    generer_pdf_profilage_dataframe(df, output_pdf_path=f'rapport_{NOM_FICHIER}.pdf')

✅ Rapport PDF généré avec succès : rapport_GMB_Fichier8.pdf


In [418]:
# records_data = []
# for record in tqdm(list_records, desc="Processing Records"):
    # d = {"id": record.num_notice}

    # for col in COLS_MATRICE:
        # if col in record2matrice:
            # d[col] = col2attr(col, record, record2matrice)
        # else:
            # d[col] = ""

    # Indexations en colonnes supplémentaires : la nomenclature I61-00-XX
    # est conservée du script initial, à vérifier avec le format du livrable final.
    # for i, item in enumerate(record.indexation):
        # res = item.replace("-- frTICO", "").replace("-- rameau", "")
        # Supprimer les suffixes « -- frTico » et « -- rameau » ajoutés à la suite des sujets. 
        # if len(res) >= 2 and res[-1] == " ": 
            # res = res[:-2]

        # if res != "" and (res[0] == "." or res[0] == "-"):
            # res = res[2:] # PQ ça ne marche pas cette affaire ? I-61-02
            # res = res.lstrip("")
        # if "Rhône" in res: print(res)
        # d[f"I61-00-{i:02d}"] = res
        # print(f"{(i+1):02d}")
        
        

In [419]:
# # Pour avoir les colonnes dans le bon ordre, sans perdre les colonnes non vides
# colonnes_indexation = sorted(col for col in matrice.columns if col.startswith("I61-00-"))
# colonnes_export = ["id"] + COLS_MATRICE + colonnes_indexation
# # La structure de base est conservée, même si certaines colonnes sont encore non renseignées
# df = matrice.reindex(columns=colonnes_export).fillna("")

# Récupérer et trier les colonnes d'indexation y compris les indexations arméniennes
## MODIFICATION : Le tri numérique conserve l'ordre des indexations au-delà de 09.
colonnes_indexation = sorted(
    (
        col for col in matrice.columns
        if col.startswith(("I61-", "I62-", "I63-"))
    ),
    key=lambda col: (col.split("-")[0], *(int(part) for part in col.split("-")[1:])),
)


# Récupérer les colonnes codes projet supplémentaires
colonnes_projet_supp = sorted(
    (
        col for col in matrice.columns
        if col.startswith("I71-")
        and col not in COLS_MATRICE
    ),
    key=lambda col: int(col.split("-")[1])
)

# Construire la liste d'export en insérant les I61-00- juste avant la section I71
colonnes_export = ["id"]
indexation_inseree = False

for col in COLS_MATRICE:
    if not indexation_inseree and col.startswith("I71-"):
        colonnes_export.extend(colonnes_indexation)
        indexation_inseree = True
    colonnes_export.append(col)

    # INSÉRONS LES CODES PROJET SUPPLÉMENTAIRES
    # APRÈS I71-03
    if col == "I71-03":
        colonnes_export.extend(colonnes_projet_supp)

# Sécurité : si aucune colonne I71 n'est trouvée dans COLS_MATRICE, on les ajoute à la fin
if not indexation_inseree:
    colonnes_export.extend(colonnes_indexation)

# Réindexation finale
df = matrice.reindex(columns=colonnes_export).fillna("")


# On fait le nettoyagz général : enlève tout point suivi d'un espace
# au début de toutes les cellules du fichier Excel
df = df.map(
    lambda valeur: re.sub(r"^\s*\.\s+", "", valeur)
    if isinstance(valeur, str) else valeur
)


### Export : à utiliser si petit fichier test

In [420]:
# df.to_excel("./matrice_lot_2.xlsx")

### Export : à utiliser pour + de 50 000 lignes

In [421]:
def clean_excel_illegal_chars(df):
    return df.map(
        lambda x: ILLEGAL_CHARACTERS_RE.sub("", str(x)) if isinstance(x, str) else x
    )

In [422]:
chunk_size = 50000

# Découpage en fichiers Excel
for i in range(0, len(df), chunk_size):
    chunk = df.iloc[i:i + chunk_size]
    chunk = clean_excel_illegal_chars(chunk) 
    chunk.to_excel(f"./RENDU_2_NOVEMBRE_{NOM_FICHIER}_partie_{i//chunk_size + 1}.xlsx", index=False)  # MODIF : nom du fichier GMB dans le nom de sortie



### Contrôles après conversion
Vérifier notamment les NNA obtenus via SRU, les colonnes vides et le format attendu de la matrice. Les anomalies SRU restent consultables dans `ERREURS_SRU` ; les valeurs en 317$a et l'adresse bibliographique en 210/214 restent à valider avec la BML.


In [423]:
# Contrôles rapides à exécuter après la conversion
print("Nombre de notices :", len(df))
print("Correspondances ISNI/NNA dans le mapping :", len(ISNI_TO_NNA))
print("ISNI demandés au SRU :", len(SRU_CACHE_NNA))
print("NNA trouvés par le SRU :", sum(bool(nna) for nna in SRU_CACHE_NNA.values()))
print("Erreurs ou formats SRU à contrôler :", len(ERREURS_SRU))
print("Colonnes exportées :", len(df.columns))
# print(ERREURS_SRU)  # À utiliser ponctuellement pour le diagnostic


Nombre de notices : 28
Correspondances ISNI/NNA dans le mapping : 1066
ISNI demandés au SRU : 2
NNA trouvés par le SRU : 0
Erreurs ou formats SRU à contrôler : 0
Colonnes exportées : 77


In [424]:
# import pandas as pd
# from os.path import join

# def verifier_champs_obligatoires(ref_path, target_path):
#     """
#     Vérifie que les colonnes marquées comme 'Obligatoire' dans le référentiel
#     sont bien remplies dans le fichier Excel cible.
#     """
#     # 1. Chargement du référentiel et identification des codes obligatoires
#     ref_df = pd.read_excel(ref_path, sheet_name='Feuil1')
    
#     # Détection des lignes où la remarque indique un caractère obligatoire
#     ref_df['Est_Obligatoire'] = ref_df['Remarque'].astype(str).str.contains('Obligatoire', case=False, na=False)
#     codes_obligatoires = ref_df.loc[ref_df['Est_Obligatoire'], 'Code colonne'].dropna().tolist()
    
#     # 2. Chargement du fichier Excel généré à auditer
#     target_df = pd.read_excel(target_path)
    
#     # 3. Vérification de la présence des données
#     erreurs = []
#     for code in codes_obligatoires:
#         if code in target_df.columns:
#             # Recherche des cellules vides (NaN ou chaînes vides)
#             lignes_vides = target_df[
#                 target_df[code].isna() | (target_df[code].astype(str).str.strip() == '')
#             ].index.tolist()
            
#             if lignes_vides:
#                 # Ajustement des numéros de ligne (index 0 + en-tête = +2 pour Excel)
#                 lignes_excel = [l + 2 for l in lignes_vides]
#                 erreurs.append(f"Colonne **{code}** : {len(lignes_vides)} valeur(s) manquante(s) aux lignes {lignes_excel}")
#         else:
#             erreurs.append(f"Colonne obligatoire **{code}** introuvable dans le fichier généré.")
            
#     # 4. Rapport final
#     if erreurs:
#         print("❌ Échec de la validation : des champs obligatoires sont vides ou absents :")
#         for err in erreurs:
#             print(f"  - {err}")
#         return False
#     else:
#         print("✅ Validation réussie : tous les champs obligatoires sont renseignés.")
#         return True
# PATH = "C:\\Users\\mathi\\OneDrive\\Bureau\\Projet Numelyo\\Projet Numelyo"
# # --- Exemple d'utilisation ---
# verifier_champs_obligatoires(join(PATH, 'GMB_Numelyo-Mapping-Unimarc-lot2.xlsm'), join(PATH, 'RENDU_2_NOVEMBRE_GMB_Fichier3_partie_1.xlsx'))